# Convolutional Autoencoder – ISIC 2020 Test Images

Trains a convolutional autoencoder on the JPEG images in `dataset/ISIC_2020_Test_Input`.

**Key design decisions**
| Parameter | Where handled | Rationale |
|---|---|---|
| Image resize (H × W) | `ISIC2020Dataset` constructor | Images stay on disk; we only resize when a sample is fetched, so the full dataset never lives in memory. |
| Batch size / workers | `DataLoader` | Standard PyTorch pattern. |
| Latent dim, channels | `ConvAutoencoder` constructor | Easy to tune. |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import glob

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from datasets import ISIC2020Dataset

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ── Parameters (edit here) ────────────────────────────────────────────────────
DATASET_DIR   = "../dataset/ISIC_2020_Test_Input"  # path relative to this notebook
IMAGE_HEIGHT  = 128          # target height after resize
IMAGE_WIDTH   = 128          # target width  after resize (can differ from height)
BATCH_SIZE    = 8
NUM_WORKERS   = 2
LATENT_DIM    = 128          # size of the bottleneck vector
BASE_CHANNELS = 32           # number of filters in the first conv layer
LEARNING_RATE = 1e-5
NUM_EPOCHS    = 20
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Target image size: {IMAGE_HEIGHT} × {IMAGE_WIDTH}")

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
class ConvAutoencoder(nn.Module):
    """
    Symmetric convolutional autoencoder.

    Parameters
    ----------
    image_height : int
    image_width  : int
    in_channels  : int   (3 for RGB)
    base_channels : int  Number of filters in the first conv layer.
                         Doubles with each down-sampling stage.
    latent_dim   : int   Dimension of the flat bottleneck vector.
    """

    def __init__(
        self,
        image_height: int  = 128,
        image_width: int   = 128,
        in_channels: int   = 3,
        base_channels: int = 32,
        latent_dim: int    = 128,
    ):
        super().__init__()
        self.image_height  = image_height
        self.image_width   = image_width
        self.base_channels = base_channels
        self.latent_dim    = latent_dim

        c1, c2, c3 = base_channels, base_channels * 2, base_channels * 4

        # ── Encoder ────────────────────────────────────────────────────────
        # Three stride-2 conv blocks; each halves H and W.
        self.encoder_conv = nn.Sequential(
            # Block 1: (N, in_channels, H,   W)   → (N, c1, H/2, W/2)
            nn.Conv2d(in_channels, c1, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c1),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 2: (N, c1, H/2, W/2) → (N, c2, H/4, W/4)
            nn.Conv2d(c1, c2, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c2),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 3: (N, c2, H/4, W/4) → (N, c3, H/8, W/8)
            nn.Conv2d(c2, c3, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c3),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # Compute the spatial size after 3 stride-2 halving operations
        self.enc_h = image_height // 8
        self.enc_w = image_width  // 8
        flat_size  = c3 * self.enc_h * self.enc_w

        self.encoder_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, latent_dim),
            nn.ReLU(inplace=True),
        )

        # ── Decoder ────────────────────────────────────────────────────────
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, flat_size),
            nn.ReLU(inplace=True),
        )

        # Three transposed-conv blocks mirror the encoder.
        self.decoder_conv = nn.Sequential(
            # Block 1: (N, c3, H/8, W/8) → (N, c2, H/4, W/4)
            nn.ConvTranspose2d(c3, c2, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c2),
            nn.ReLU(inplace=True),

            # Block 2: (N, c2, H/4, W/4) → (N, c1, H/2, W/2)
            nn.ConvTranspose2d(c2, c1, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True),

            # Block 3: (N, c1, H/2, W/2) → (N, in_channels, H, W)
            nn.ConvTranspose2d(c1, in_channels, 4, stride=2, padding=1, bias=False),
            nn.Tanh(),  # output in [-1, 1] to match Normalize(0.5, 0.5)
        )

    # ------------------------------------------------------------------
    def encode(self, x):
        return self.encoder_fc(self.encoder_conv(x))

    def decode(self, z):
        h = self.decoder_fc(z)
        h = h.view(h.size(0), self.base_channels * 4, self.enc_h, self.enc_w)
        return self.decoder_conv(h)

    def forward(self, x):
        return self.decode(self.encode(x))

In [ ]:
# ── Dataset & DataLoader ──────────────────────────────────────────────────────
dataset = ISIC2020Dataset(
    root_dir      = DATASET_DIR,
    target_height = IMAGE_HEIGHT,
    target_width  = IMAGE_WIDTH,
)

loader = DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    pin_memory  = DEVICE == "cuda",
)

print(f"Total samples : {len(dataset)}")
print(f"Batches/epoch : {len(loader)}")

In [ ]:
# ── Visualise a batch (sanity check) ─────────────────────────────────────────
sample_batch = next(iter(loader))          # shape: (B, 3, IMAGE_HEIGHT, IMAGE_WIDTH)
print("Batch shape:", sample_batch.shape)
print("Value range:", sample_batch.min().item(), "→", sample_batch.max().item())

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i, ax in enumerate(axes):
    img = sample_batch[i].permute(1, 2, 0).numpy()  # C,H,W → H,W,C
    img = (img * 0.5) + 0.5                          # un-normalise to [0,1]
    img = img.clip(0, 1)
    ax.imshow(img)
    ax.axis("off")
plt.suptitle("Sample images from DataLoader", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Model / optimiser / loss ──────────────────────────────────────────────────
model = ConvAutoencoder(
    image_height  = IMAGE_HEIGHT,
    image_width   = IMAGE_WIDTH,
    in_channels   = 3,
    base_channels = BASE_CHANNELS,
    latent_dim    = LATENT_DIM,
).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
from tqdm import tqdm

epoch_losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, desc=f"Epoch [{epoch:>3}/{NUM_EPOCHS}]", leave=True)
    for i, batch in enumerate(pbar, start=1):
        batch = batch.to(DEVICE, non_blocking=True)

        recon = model(batch)
        loss  = criterion(recon, batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch.size(0)
        avg_loss = running_loss / (i * BATCH_SIZE)
        pbar.set_postfix(avg_loss=f"{avg_loss:.6f}")

    epoch_loss = running_loss / len(dataset)
    epoch_losses.append(epoch_loss)

print("\nTraining complete.")

In [ ]:
# ── Loss curve ────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS + 1), epoch_losses, marker="o", linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# ── Reconstruction quality ────────────────────────────────────────────────────
model.eval()
n_show = 6

with torch.no_grad():
    sample = next(iter(loader))[:n_show].to(DEVICE)
    recon  = model(sample).cpu()
    sample = sample.cpu()

def unnorm(t):
    return ((t * 0.5) + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, n_show, figsize=(15, 5))
for i in range(n_show):
    axes[0, i].imshow(unnorm(sample[i]))
    axes[0, i].axis("off")
    axes[0, i].set_title("Original")

    axes[1, i].imshow(unnorm(recon[i]))
    axes[1, i].axis("off")
    axes[1, i].set_title("Reconstructed")

plt.suptitle("Original vs. Reconstructed images", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save checkpoint ───────────────────────────────────────────────────────────
CHECKPOINT_PATH = "autoencoder_isic.pth"

torch.save({
    "epoch"       : NUM_EPOCHS,
    "model_state" : model.state_dict(),
    "optim_state" : optimizer.state_dict(),
    "config": {
        "image_height" : IMAGE_HEIGHT,
        "image_width"  : IMAGE_WIDTH,
        "base_channels": BASE_CHANNELS,
        "latent_dim"   : LATENT_DIM,
    },
    "epoch_losses" : epoch_losses,
}, CHECKPOINT_PATH)

print(f"Checkpoint saved to '{CHECKPOINT_PATH}'")